In [ ]:
import numpy as np
from scipy.integrate import odeint
from typing import Dict, Optional


def synthesize_periodic(n: int, freq: float = 1.0, dt: float = 0.01) -> np.ndarray:
    """Pure periodic: single sinusoid."""
    t = np.arange(n) * dt
    return np.sin(2 * np.pi * freq * t)


def synthesize_quasi_periodic(n: int, dt: float = 0.01) -> np.ndarray:
    """Quasi-periodic: sum of incommensurate frequencies (e.g. sqrt(2), sqrt(3))."""
    t = np.arange(n) * dt
    return np.sin(2 * np.pi * np.sqrt(2) * t) + 0.5 * np.sin(2 * np.pi * np.sqrt(3) * t)


def lorenz(state: np.ndarray, t: float, sigma: float, rho: float, beta: float) -> np.ndarray:
    """Lorenz system derivatives."""
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return np.array([dx, dy, dz])


def synthesize_chaotic_lorenz_x(n: int, dt: float = 0.01, sigma: float = 10.0,
                                rho: float = 28.0, beta: float = 8/3) -> np.ndarray:
    """Chaotic: x(t) from Lorenz system."""
    t = np.linspace(0, (n - 1) * dt, n)
    state0 = [1.0, 1.0, 1.0]
    sol = odeint(lorenz, state0, t, args=(sigma, rho, beta))
    return sol[:, 0]  # x(t)


def synthesize_noise(n: int, scale: float = 1.0, seed: Optional[int] = None) -> np.ndarray:
    """White Gaussian noise."""
    rng = np.random.default_rng(seed)
    return rng.standard_normal(n) * scale


def delay_embed(x: np.ndarray, m: int, tau: int) -> np.ndarray:
    """Build delay embedding: rows are [x(t), x(t-tau), ..., x(t-(m-1)*tau)]."""
    n = len(x)
    L = n - (m - 1) * tau
    if L <= 0:
        raise ValueError("Series too short for m and tau: need n > (m-1)*tau")
    rows = [x[i: i + L] for i in range(0, m * tau, tau)]
    return np.column_stack(rows)


def build_embeddings(
    n: int = 2000,
    m: int = 3,
    tau: int = 10,
    dt: float = 0.01,
    noise_scale: float = 1.0,
    seed: Optional[int] = 42,
) -> Dict[str, np.ndarray]:
    """
    Synthesize 4 canonical time series and return delay embeddings.

    Parameters
    ----------
    n : int
        Length of each time series.
    m : int
        Embedding dimension.
    tau : int
        Delay (in samples).
    dt : float
        Time step for synthetic series.
    noise_scale : float
        Scale of white noise.
    seed : int, optional
        Random seed for noise (and Lorenz if desired).

    Returns
    -------
    dict
        Keys: 'periodic', 'quasi_periodic', 'chaotic', 'noise'.
        Values: 2D arrays of shape (L, m) with L = n - (m-1)*tau.
    """
    periodic = synthesize_periodic(n, dt=dt)
    quasi_periodic = synthesize_quasi_periodic(n, dt=dt)
    chaotic = synthesize_chaotic_lorenz_x(n, dt=dt)
    noise = synthesize_noise(n, scale=noise_scale, seed=seed)

    embeddings = {
        "periodic": delay_embed(periodic, m, tau),
        "quasi_periodic": delay_embed(quasi_periodic, m, tau),
        "chaotic": delay_embed(chaotic, m, tau),
        "noise": delay_embed(noise, m, tau),
    }
    return embeddings




In [ ]:
# Stage 0: Parameters
# These must match the embedding params used in ph_summary_from_window
n = 2000
dt = 0.01
m, tau = 3, 10        # global delay embedding params
m_win, tau_win = 3, 5  # embedding params for sliding-window PH
L = 50                 # sliding window length

# Stage 1: Synthesize time series
series = {
    'periodic':      synthesize_periodic(n, dt=dt),
    'quasi_periodic': synthesize_quasi_periodic(n, dt=dt),
    'chaotic':        synthesize_chaotic_lorenz_x(n, dt=dt),
    'noise':          synthesize_noise(n, seed=42),
}
print('Series synthesized:', {k: len(v) for k, v in series.items()})
print(f'Window: L={L}, m_win={m_win}, tau_win={tau_win}')
print(f'Window embedding points: {L - (m_win - 1) * tau_win}')

In [ ]:
import numpy as np
from ripser import ripser
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --------------- Stage 0: Delay embedding ----------------
def delay_embed_window(x, m, tau):
    n_pts = len(x) - (m - 1) * tau
    if n_pts <= 0:
        return np.zeros((0, m))
    rows = [x[i:i+n_pts] for i in range(0, m*tau, tau)]
    return np.column_stack(rows)

# --------------- Stage 1: PH summaries ----------------
def _ph_summary_dim(kind, **kwargs):
    """Expected output length for ph_summary_from_window for given kind (for early-return fallback)."""
    if kind == "basic":
        return 4
    if kind == "landscape":
        return kwargs.get("n_levels", 5) * kwargs.get("n_pts", 20)
    if kind == "betti":
        return kwargs.get("n_bins", 10)
    return 4

def ph_summary_from_window(window, m, tau, kind="basic", **kwargs):
    X = delay_embed_window(window, m, tau)
    if len(X) < 4:
        return np.zeros(_ph_summary_dim(kind, **kwargs))
    dgms = ripser(X, maxdim=1)["dgms"]
    h1 = dgms[1][np.isfinite(dgms[1][:,1])]
    if len(h1) == 0:
        return np.zeros(_ph_summary_dim(kind, **kwargs))
    
    if kind == "basic":
        pers = h1[:,1]-h1[:,0]
        return np.array([pers.mean(), pers.max(), len(pers), pers.std() if len(pers)>1 else 0.0])
    
    elif kind == "landscape":
        n_levels = kwargs.get("n_levels",5)
        n_pts = kwargs.get("n_pts",20)
        birth, death = h1[:,0], h1[:,1]
        x_grid = np.linspace(0, max(death), n_pts)
        land = np.zeros((n_levels, n_pts))
        for k in range(n_levels):
            for b,d in zip(birth,death):
                mid, half = (b+d)/2, (d-b)/2
                land[k] += np.maximum(0, half - np.abs(x_grid - mid))
        return land.flatten()
    
    elif kind == "betti":
        n_bins = kwargs.get("n_bins",10)
        f_min, f_max = h1[:,0].min(), h1[:,1].max()
        edges = np.linspace(f_min, f_max, n_bins+1)
        bc = np.zeros(n_bins)
        for b,d in h1:
            for i in range(n_bins):
                if edges[i]<d and b<edges[i+1]:
                    bc[i]+=1
        return bc
    else:
        raise ValueError(f"Unknown PH summary kind: {kind}")

# --------------- Stage 2: Build windows & features ---------------
def build_dataset(series, L, m, tau, ph_kind="basic", include_ph=True):
    X_raw, X_ph, y = [], [], []
    for ts_name, ts in series.items():
        for i in range(len(ts)-L):
            w = ts[i:i+L]
            X_raw.append(w)
            if include_ph:
                X_ph.append(ph_summary_from_window(w, m, tau, kind=ph_kind))
            y.append(ts[i+L])
    X_raw = np.array(X_raw)
    X_ph = np.array(X_ph) if include_ph else None
    y = np.array(y)
    if include_ph:
        X = np.hstack([X_raw, X_ph])
    else:
        X = X_raw
    return X_raw, X_ph, X, y

# --------------- Stage 3: Train/test split ----------------
def temporal_split(series, L, m, tau, ph_kind='basic', train_frac=0.8, include_ph=True):
    X_raw_tr, X_raw_te, X_comb_tr, X_comb_te, y_tr, y_te = [],[],[],[],[],[]
    for ts_name, ts in series.items():
        split_idx = int(len(ts)*train_frac)
        for i in range(split_idx-L):
            w = ts[i:i+L]
            X_raw_tr.append(w)
            if include_ph:
                X_comb_tr.append(np.hstack([w, ph_summary_from_window(w, m, tau, kind=ph_kind)]))
            y_tr.append(ts[i+L])
        for i in range(split_idx, len(ts)-L):
            w = ts[i:i+L]
            X_raw_te.append(w)
            if include_ph:
                X_comb_te.append(np.hstack([w, ph_summary_from_window(w, m, tau, kind=ph_kind)]))
            y_te.append(ts[i+L])
    X_raw_tr, X_raw_te = np.array(X_raw_tr), np.array(X_raw_te)
    y_tr, y_te = np.array(y_tr), np.array(y_te)
    if include_ph:
        X_comb_tr, X_comb_te = np.array(X_comb_tr), np.array(X_comb_te)
    else:
        X_comb_tr, X_comb_te = X_raw_tr, X_raw_te
    return X_raw_tr, X_raw_te, X_comb_tr, X_comb_te, y_tr, y_te

# --------------- Stage 4: Scale features ----------------
def scale_features(X_raw_tr, X_raw_te, X_comb_tr, X_comb_te):
    scaler_raw = StandardScaler()
    scaler_ph = StandardScaler()
    X_raw_tr_s = scaler_raw.fit_transform(X_raw_tr)
    X_raw_te_s = scaler_raw.transform(X_raw_te)
    X_comb_tr_s = scaler_ph.fit_transform(X_comb_tr)
    X_comb_te_s = scaler_ph.transform(X_comb_te)
    return X_raw_tr_s, X_raw_te_s, X_comb_tr_s, X_comb_te_s

# --------------- Stage 5: Linear model ----------------
def train_linear(X_tr, y_tr):
    lm = LinearRegression().fit(X_tr, y_tr)
    return lm

def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return mse, mae

In [ ]:
# Stage 5: Prediction experiment
# Compare linear regression with/without PH features (basic, landscape, betti)
system_order = ['periodic', 'quasi_periodic', 'chaotic', 'noise']

for ph_kind in ['basic', 'landscape', 'betti']:
    # Build temporal splits with PH features
    X_raw_tr, X_raw_te, X_comb_tr, X_comb_te, y_tr, y_te = temporal_split(
        series, L, m_win, tau_win, ph_kind=ph_kind, include_ph=True
    )
    X_raw_tr2, X_raw_te2, _, _, y_tr2, y_te2 = temporal_split(
        series, L, m_win, tau_win, ph_kind=ph_kind, include_ph=False
    )

    scaler_r = StandardScaler()
    scaler_c = StandardScaler()
    X_raw_tr_s = scaler_r.fit_transform(X_raw_tr)
    X_raw_te_s = scaler_r.transform(X_raw_te)
    X_comb_tr_s = scaler_c.fit_transform(X_comb_tr)
    X_comb_te_s = scaler_c.transform(X_comb_te)

    lm_r = train_linear(X_raw_tr_s, y_tr)
    lm_c = train_linear(X_comb_tr_s, y_tr)

    mse_r, mae_r = evaluate(y_te, lm_r.predict(X_raw_te_s))
    mse_c, mae_c = evaluate(y_te, lm_c.predict(X_comb_te_s))

    delta = (mse_c - mse_r) / mse_r * 100
    print(f'{ph_kind:>10}: raw MSE={mse_r:.4f} | +PH MSE={mse_c:.4f} | delta={delta:+.2f}%')

In [ ]:
# Stage 5b: Noise Invariance Analysis
# Adding Gaussian noise to periodic signal, tracking PH stability.
from persim import wasserstein

def delay_embed(x, m, tau):
    n = len(x)
    L_ = n - (m - 1) * tau
    rows = [x[i: i + L_] for i in range(0, m * tau, tau)]
    return np.column_stack(rows)

def get_h1(X, subsample=500):
    if len(X) > subsample:
        idx = np.random.default_rng(0).choice(len(X), subsample, replace=False)
        X = X[idx]
    dgms = ripser(X, maxdim=1)["dgms"]
    h1 = dgms[1]
    return h1[np.isfinite(h1[:,1])] if len(h1) > 0 else np.zeros((0, 2))

periodic = series['periodic']
X_clean = delay_embed(periodic, m, tau)
h1_clean = get_h1(X_clean)

rng_ni = np.random.default_rng(99)
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
print('noise_sigma | W_dist_from_clean | top_persistence | n_H1_bars')
print('-' * 60)
for sigma in noise_levels:
    noisy = periodic + rng_ni.standard_normal(n) * sigma
    h1_n = get_h1(delay_embed(noisy, m, tau))
    dist = wasserstein(h1_clean, h1_n) if len(h1_clean) and len(h1_n) else float('nan')
    top_p = (h1_n[:,1]-h1_n[:,0]).max() if len(h1_n) else 0.0
    print(f'  sigma={sigma:.2f}   | {dist:>17.3f} | {top_p:>15.4f} | {len(h1_n):>9}')